# Single-Factor Vasicek Model

This notebook calibrates a separate univariate Vasicek model to each US Treasury maturity:

- 6M, 1Y, 2Y, 3Y, 5Y, 7Y and 10Y

The Vasicek process is:

$$
dr_t = \kappa(\theta-r_t)dt+\sigma dW_t
$$

where $\kappa$ is the speed of mean reversion, $\theta$ is the long-run mean and $\sigma$ is the instantaneous annual volatility.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import PercentFormatter
from scipy.optimize import minimize
from scipy.stats import norm, probplot

In [ ]:
df = pd.read_csv("../data/processed/cleaned_yield_curve.csv")

maturities = ["6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y"]

## Exact Vasicek transition

For a time step $\Delta t$:

$$
r_{t+\Delta t}
=
\theta+(r_t-\theta)e^{-\kappa\Delta t}
+
\sigma\sqrt{\frac{1-e^{-2\kappa\Delta t}}{2\kappa}}Z,
\qquad Z\sim\mathcal N(0,1)
$$

Therefore:

$$
\mathbb E[r_{t+\Delta t}\mid r_t]
=
\theta+(r_t-\theta)e^{-\kappa\Delta t}
$$

and

$$
\operatorname{Var}(r_{t+\Delta t}\mid r_t)
=
\frac{\sigma^2}{2\kappa}\left(1-e^{-2\kappa\Delta t}\right).
$$

In [ ]:
def vasicek_negative_log_likelihood(parameters, r_t, r_next, delta_t):
    log_kappa, theta, log_sigma = parameters
    kappa = np.exp(log_kappa)
    sigma = np.exp(log_sigma)

    phi = np.exp(-kappa * delta_t)
    conditional_mean = theta + (r_t - theta) * phi
    conditional_variance = sigma**2 * (1 - phi**2) / (2 * kappa)

    if np.any(conditional_variance <= 0):
        return np.inf

    residuals = r_next - conditional_mean
    log_likelihood_terms = -0.5 * (
        np.log(2 * np.pi * conditional_variance)
        + residuals**2 / conditional_variance
    )

    return -np.sum(log_likelihood_terms)

In [ ]:
def fit_vasicek_mle(data, maturity, date_column="Date", rates_are_percent=True):
    model_data = data[[date_column, maturity]].copy()
    model_data[date_column] = pd.to_datetime(model_data[date_column], errors="coerce")
    model_data[maturity] = pd.to_numeric(model_data[maturity], errors="coerce")

    model_data = (
        model_data.dropna()
        .sort_values(date_column)
        .drop_duplicates(subset=date_column)
        .reset_index(drop=True)
    )

    model_data["Rate"] = model_data[maturity] / 100 if rates_are_percent else model_data[maturity]
    model_data["Next_Date"] = model_data[date_column].shift(-1)
    model_data["Next_Rate"] = model_data["Rate"].shift(-1)
    model_data["Calendar_Days"] = (model_data["Next_Date"] - model_data[date_column]).dt.days
    model_data["Delta_t"] = model_data["Calendar_Days"] / 365.25

    transitions = (
        model_data[[date_column, "Next_Date", "Rate", "Next_Rate", "Calendar_Days", "Delta_t"]]
        .dropna()
        .query("Delta_t > 0")
        .rename(columns={"Rate": "r_t", "Next_Rate": "r_next"})
        .reset_index(drop=True)
    )

    r_t = transitions["r_t"].to_numpy(dtype=float)
    r_next = transitions["r_next"].to_numpy(dtype=float)
    delta_t = transitions["Delta_t"].to_numpy(dtype=float)

    kappa_initial = 0.50
    theta_initial = r_t.mean()
    sigma_initial = np.sqrt(np.sum((r_next - r_t)**2) / np.sum(delta_t))

    result = minimize(
        vasicek_negative_log_likelihood,
        x0=np.array([np.log(kappa_initial), theta_initial, np.log(sigma_initial)]),
        args=(r_t, r_next, delta_t),
        method="L-BFGS-B",
        bounds=[
            (np.log(1e-6), np.log(10)),
            (-0.10, 0.20),
            (np.log(1e-6), np.log(1)),
        ],
    )

    log_kappa, theta, log_sigma = result.x
    kappa = np.exp(log_kappa)
    sigma = np.exp(log_sigma)

    phi = np.exp(-kappa * delta_t)
    fitted_mean = theta + (r_t - theta) * phi
    fitted_variance = sigma**2 * (1 - phi**2) / (2 * kappa)
    fitted_sd = np.sqrt(fitted_variance)
    residuals = r_next - fitted_mean
    standardized_residuals = residuals / fitted_sd

    transitions["Fitted_Mean"] = fitted_mean
    transitions["Fitted_SD"] = fitted_sd
    transitions["Residual"] = residuals
    transitions["Standardized_Residual"] = standardized_residuals

    parameters = {
        "Maturity": maturity,
        "Number_of_Observations": len(model_data),
        "Kappa": kappa,
        "Theta": theta,
        "Theta_Percent": theta * 100,
        "Sigma": sigma,
        "Sigma_Percent": sigma * 100,
        "Half_Life_Years": np.log(2) / kappa,
        "Log_Likelihood": -result.fun,
        "Optimization_Success": result.success,
        "Latest_Rate": model_data["Rate"].iloc[-1],
        "Latest_Date": model_data[date_column].iloc[-1],
        "Residual_Mean": residuals.mean(),
        "Standardized_Residual_Mean": standardized_residuals.mean(),
        "Standardized_Residual_SD": standardized_residuals.std(ddof=0),
    }

    return parameters, transitions

## Calibrate all maturities

In [ ]:
all_parameter_results = []
all_validation_results = {}

for maturity in maturities:
    parameters, validation = fit_vasicek_mle(df, maturity)
    all_parameter_results.append(parameters)
    all_validation_results[maturity] = validation

vasicek_parameter_table = pd.DataFrame(all_parameter_results)

In [ ]:
parameter_display = vasicek_parameter_table[
    [
        "Maturity",
        "Number_of_Observations",
        "Kappa",
        "Theta_Percent",
        "Sigma_Percent",
        "Half_Life_Years",
        "Log_Likelihood",
        "Optimization_Success",
    ]
].copy()

parameter_display[
    ["Kappa", "Theta_Percent", "Sigma_Percent", "Half_Life_Years", "Log_Likelihood"]
] = parameter_display[
    ["Kappa", "Theta_Percent", "Sigma_Percent", "Half_Life_Years", "Log_Likelihood"]
].round(4)

parameter_display

In [ ]:
residual_diagnostics = vasicek_parameter_table[
    [
        "Maturity",
        "Residual_Mean",
        "Standardized_Residual_Mean",
        "Standardized_Residual_SD",
    ]
].copy()

residual_diagnostics.iloc[:, 1:] = residual_diagnostics.iloc[:, 1:].round(4)
residual_diagnostics

In [ ]:
probplot(
    all_validation_results["6M"]["Standardized_Residual"],
    dist="norm",
    plot=plt,
)
plt.title("Q–Q Plot of 6M Standardized Residuals")
plt.grid(alpha=0.3)
plt.show()

## Simulate one-year Vasicek paths

In [ ]:
def simulate_vasicek(
    kappa,
    theta,
    sigma,
    initial_rate,
    simulation_horizon_years=1,
    steps_per_year=252,
    number_of_simulations=10000,
    seed=42,
):
    number_of_steps = int(simulation_horizon_years * steps_per_year)
    delta_t = 1 / steps_per_year

    phi = np.exp(-kappa * delta_t)
    conditional_sd = np.sqrt(sigma**2 * (1 - phi**2) / (2 * kappa))

    rng = np.random.default_rng(seed)
    paths = np.empty((number_of_steps + 1, number_of_simulations))
    paths[0] = initial_rate

    for step in range(1, number_of_steps + 1):
        paths[step] = (
            theta
            + (paths[step - 1] - theta) * phi
            + conditional_sd * rng.normal(size=number_of_simulations)
        )

    simulation_times = np.linspace(0, simulation_horizon_years, number_of_steps + 1)
    paths = pd.DataFrame(paths, index=simulation_times)
    paths.index.name = "Time_In_Years"

    terminal_rates = paths.iloc[-1]

    return {
        "Paths": paths,
        "Terminal_Rates": terminal_rates,
        "Simulated_Changes": terminal_rates - initial_rate,
    }

In [ ]:
all_simulation_results = {}

for maturity_number, maturity in enumerate(maturities):
    parameter_row = vasicek_parameter_table.loc[
        vasicek_parameter_table["Maturity"] == maturity
    ].iloc[0]

    all_simulation_results[maturity] = simulate_vasicek(
        kappa=parameter_row["Kappa"],
        theta=parameter_row["Theta"],
        sigma=parameter_row["Sigma"],
        initial_rate=parameter_row["Latest_Rate"],
        seed=42 + maturity_number,
    )

In [ ]:
simulated_change_summary = pd.DataFrame(
    [
        {
            "Maturity": maturity,
            "Mean_Change": result["Simulated_Changes"].mean(),
            "Standard_Deviation": result["Simulated_Changes"].std(ddof=1),
            "0.5th_Percentile": result["Simulated_Changes"].quantile(0.005),
            "Median": result["Simulated_Changes"].median(),
            "99.5th_Percentile": result["Simulated_Changes"].quantile(0.995),
        }
        for maturity, result in all_simulation_results.items()
    ]
)

change_columns = [
    "Mean_Change",
    "Standard_Deviation",
    "0.5th_Percentile",
    "Median",
    "99.5th_Percentile",
]

simulated_change_summary_bps = simulated_change_summary.copy()
simulated_change_summary_bps[change_columns] = (
    simulated_change_summary_bps[change_columns] * 10000
).round(2)

simulated_change_summary_bps

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 18), sharex=True)
axes = axes.flatten()

for index, maturity in enumerate(maturities):
    ax = axes[index]
    paths = all_simulation_results[maturity]["Paths"]
    parameter_row = vasicek_parameter_table.loc[
        vasicek_parameter_table["Maturity"] == maturity
    ].iloc[0]

    ax.plot(paths.index, paths.iloc[:, :100], linewidth=0.5, alpha=0.25)
    ax.axhline(parameter_row["Theta"], linestyle="--", linewidth=2, label="Long-run mean")
    ax.axhline(parameter_row["Latest_Rate"], linestyle=":", linewidth=2, label="Starting rate")
    ax.set_title(f"{maturity} Vasicek Simulated Paths")
    ax.set_xlabel("Time in years")
    ax.set_ylabel("Yield")
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
    ax.grid(alpha=0.3)
    ax.legend()

fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()

## Compare simulated and historical one-year changes

In [ ]:
def calculate_historical_one_year_changes(
    data,
    maturity,
    date_column="Date",
    rates_are_percent=True,
    tolerance_days=7,
):
    historical_data = data[[date_column, maturity]].copy()
    historical_data[date_column] = pd.to_datetime(historical_data[date_column], errors="coerce")
    historical_data[maturity] = pd.to_numeric(historical_data[maturity], errors="coerce")

    historical_data = (
        historical_data.dropna()
        .sort_values(date_column)
        .drop_duplicates(subset=date_column)
        .reset_index(drop=True)
    )

    historical_data["Rate"] = (
        historical_data[maturity] / 100 if rates_are_percent else historical_data[maturity]
    )

    starts = historical_data[[date_column, "Rate"]].copy()
    starts["Target_Date"] = starts[date_column] + pd.DateOffset(years=1)

    future = historical_data[[date_column, "Rate"]].rename(
        columns={date_column: "Future_Date", "Rate": "Future_Rate"}
    )

    matched = pd.merge_asof(
        starts.sort_values("Target_Date"),
        future.sort_values("Future_Date"),
        left_on="Target_Date",
        right_on="Future_Date",
        direction="nearest",
        tolerance=pd.Timedelta(days=tolerance_days),
    ).dropna(subset=["Future_Rate"])

    matched["Historical_1Y_Change"] = matched["Future_Rate"] - matched["Rate"]
    matched["Maturity"] = maturity

    return matched.reset_index(drop=True)

In [ ]:
all_historical_change_results = {
    maturity: calculate_historical_one_year_changes(df, maturity)
    for maturity in maturities
}

historical_change_summary = pd.DataFrame(
    [
        {
            "Maturity": maturity,
            "Number_of_Changes": result["Historical_1Y_Change"].count(),
            "Mean_Change": result["Historical_1Y_Change"].mean(),
            "Standard_Deviation": result["Historical_1Y_Change"].std(ddof=1),
            "0.5th_Percentile": result["Historical_1Y_Change"].quantile(0.005),
            "Median": result["Historical_1Y_Change"].median(),
            "99.5th_Percentile": result["Historical_1Y_Change"].quantile(0.995),
        }
        for maturity, result in all_historical_change_results.items()
    ]
)

historical_change_summary_bps = historical_change_summary.copy()
historical_change_summary_bps[change_columns] = (
    historical_change_summary_bps[change_columns] * 10000
).round(2)

historical_change_summary_bps

In [ ]:
tail_comparison = historical_change_summary[
    [
        "Maturity",
        "Number_of_Changes",
        "Standard_Deviation",
        "0.5th_Percentile",
        "99.5th_Percentile",
    ]
].merge(
    simulated_change_summary[
        ["Maturity", "Standard_Deviation", "0.5th_Percentile", "99.5th_Percentile"]
    ],
    on="Maturity",
    suffixes=("_Historical", "_Vasicek"),
)

tail_comparison["Lower_Tail_Difference"] = (
    tail_comparison["0.5th_Percentile_Vasicek"]
    - tail_comparison["0.5th_Percentile_Historical"]
)
tail_comparison["Upper_Tail_Difference"] = (
    tail_comparison["99.5th_Percentile_Vasicek"]
    - tail_comparison["99.5th_Percentile_Historical"]
)

comparison_columns = [
    "Standard_Deviation_Historical",
    "Standard_Deviation_Vasicek",
    "0.5th_Percentile_Historical",
    "0.5th_Percentile_Vasicek",
    "99.5th_Percentile_Historical",
    "99.5th_Percentile_Vasicek",
    "Lower_Tail_Difference",
    "Upper_Tail_Difference",
]

tail_comparison_bps = tail_comparison.copy()
tail_comparison_bps[comparison_columns] = (
    tail_comparison_bps[comparison_columns] * 10000
).round(2)

tail_comparison_bps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x_positions = np.arange(len(maturities))
bar_width = 0.35

axes[0].bar(
    x_positions - bar_width / 2,
    tail_comparison_bps["0.5th_Percentile_Historical"],
    width=bar_width,
    label="Historical",
)
axes[0].bar(
    x_positions + bar_width / 2,
    tail_comparison_bps["0.5th_Percentile_Vasicek"],
    width=bar_width,
    label="Vasicek",
)
axes[0].set_title("Historical vs Vasicek 0.5th Percentile")
axes[0].set_ylabel("One-year change in basis points")
axes[0].set_xticks(x_positions, maturities)
axes[0].axhline(0, linewidth=0.8)
axes[0].grid(axis="y", alpha=0.3)
axes[0].legend()

axes[1].bar(
    x_positions - bar_width / 2,
    tail_comparison_bps["99.5th_Percentile_Historical"],
    width=bar_width,
    label="Historical",
)
axes[1].bar(
    x_positions + bar_width / 2,
    tail_comparison_bps["99.5th_Percentile_Vasicek"],
    width=bar_width,
    label="Vasicek",
)
axes[1].set_title("Historical vs Vasicek 99.5th Percentile")
axes[1].set_ylabel("One-year change in basis points")
axes[1].set_xticks(x_positions, maturities)
axes[1].axhline(0, linewidth=0.8)
axes[1].grid(axis="y", alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()